# Traductor Inglés → Español con Transformer (desde cero)

**Curso:** Tópicos Avanzados en Machine Learning — Proyecto de Neural Networks

Este notebook implementa, **desde cero** (sin modelos ni pesos pre-entrenados, sin librerías de NMT de alto nivel), un modelo
**Transformer** encoder-decoder para traducción automática inglés→español, siguiendo la arquitectura de
*Attention Is All You Need* (Vaswani et al., 2017). Solo se usan primitivas de `torch.nn` (Linear, Embedding, LayerNorm, Dropout);
la atención multi-cabeza, el positional encoding, el encoder y el decoder están escritos a mano.

**Diseñado para correr en Google Colab con GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU).

## Contenido
1. Setup e imports
2. Descarga y extracción de datos (Tatoeba / Anki eng-spa)
3. Limpieza, tokenización y vocabulario (desde cero)
4. Dataset y DataLoader
5. Arquitectura Transformer (desde cero)
6. Entrenamiento
7. Inferencia (decodificación greedy)
8. Evaluación en conjunto de test (BLEU, ≥100 ejemplos, casos buenos y casos donde falla)
9. Guardado de pesos y vocabularios (para el notebook de inferencia)
10. Reporte y conclusiones

## 1. Setup e imports

In [ ]:
import os, re, json, math, time, random, zipfile, urllib.request
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import pandas as pd

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Descarga y extracción de datos

Usamos el corpus de pares de oraciones inglés-español de **Tatoeba**, distribuido por el proyecto
[ManyThings.org / Anki](http://www.manythings.org/anki/) (`spa-eng.zip`). Cada línea del archivo `spa.txt`
tiene el formato `english TAB spanish TAB attribution`.

In [ ]:
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# manythings.org bloquea peticiones sin User-Agent de navegador (403).
# Si aun así falla (o cambia de URL), se usa como respaldo el mismo
# archivo (spa.txt de Tatoeba/Anki) alojado por TensorFlow.
DATA_URLS = [
    'https://www.manythings.org/anki/spa-eng.zip',
    'http://www.manythings.org/anki/spa-eng.zip',
    'https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip',
]
zip_path = os.path.join(DATA_DIR, 'spa-eng.zip')
txt_path = os.path.join(DATA_DIR, 'spa.txt')

def download_dataset():
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    last_err = None
    for url in DATA_URLS:
        try:
            print(f'Descargando dataset desde {url} ...')
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp, open(zip_path, 'wb') as f:
                f.write(resp.read())
            return
        except Exception as e:
            print(f'  falló ({e}), probando siguiente fuente...')
            last_err = e
    raise RuntimeError('No se pudo descargar el dataset de ninguna fuente') from last_err

if not os.path.exists(txt_path):
    if not os.path.exists(zip_path):
        download_dataset()
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    # el mirror de TensorFlow extrae a data/spa-eng/spa.txt en vez de data/spa.txt
    alt_path = os.path.join(DATA_DIR, 'spa-eng', 'spa.txt')
    if not os.path.exists(txt_path) and os.path.exists(alt_path):
        txt_path = alt_path
    print('Dataset extraído en', txt_path)
else:
    print('Dataset ya existe en', txt_path)

## 3. Limpieza, tokenización y vocabulario (desde cero)

No usamos tokenizadores pre-entrenados (ni BPE de librerías externas): la normalización de texto y la
construcción del vocabulario son propias. Se separa la puntuación, se pasa a minúsculas y se filtran
caracteres fuera del alfabeto español/inglés.

In [ ]:
MAX_LEN = 20      # longitud máxima de oración (en tokens) que se conserva
MIN_FREQ = 2      # frecuencia mínima para que una palabra entre al vocabulario

PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'

def normalize_text(s):
    s = s.strip().lower()
    s = re.sub(r"([.!?¿¡,])", r" \1 ", s)
    s = re.sub(r"[^a-zñáéíóúü¿¡.!?, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

pairs = []
with open(txt_path, encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip('\n').split('\t')
        if len(parts) < 2:
            continue
        en, es = parts[0], parts[1]
        en_n, es_n = normalize_text(en), normalize_text(es)
        if not en_n or not es_n:
            continue
        if len(en_n.split()) > MAX_LEN or len(es_n.split()) > MAX_LEN:
            continue
        pairs.append((en_n, es_n))

print(f'Total de pares después de limpiar: {len(pairs)}')
print('Ejemplo:', pairs[0])

In [ ]:
class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}

    def build(self, sentences):
        counter = Counter()
        for s in sentences:
            counter.update(s.split())
        specials = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        words = specials + [w for w, c in counter.items() if c >= self.min_freq]
        self.word2idx = {w: i for i, w in enumerate(words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    def encode(self, sentence, add_sos_eos=True):
        ids = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in sentence.split()]
        if add_sos_eos:
            ids = [self.word2idx[SOS_TOKEN]] + ids + [self.word2idx[EOS_TOKEN]]
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK_TOKEN)
            if w == EOS_TOKEN:
                break
            if w in (SOS_TOKEN, PAD_TOKEN):
                continue
            words.append(w)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

### División train / val / test

Se separan al menos 100 pares para el conjunto de test (usado en la sección de evaluación) y otros 100+
para validación durante el entrenamiento. El vocabulario se construye **solo con el conjunto de train**
para evitar fuga de información (*data leakage*).

In [ ]:
random.shuffle(pairs)

n = len(pairs)
n_test = max(100, int(n * 0.02))
n_val = max(100, int(n * 0.02))

test_pairs = pairs[:n_test]
val_pairs = pairs[n_test:n_test + n_val]
train_pairs = pairs[n_test + n_val:]

print(f'Train: {len(train_pairs)}  Val: {len(val_pairs)}  Test: {len(test_pairs)}')

src_vocab = Vocab(min_freq=MIN_FREQ)
src_vocab.build([p[0] for p in train_pairs])
tgt_vocab = Vocab(min_freq=MIN_FREQ)
tgt_vocab.build([p[1] for p in train_pairs])

print(f'Vocabulario inglés: {len(src_vocab)} palabras')
print(f'Vocabulario español: {len(tgt_vocab)} palabras')

## 4. Dataset y DataLoader

In [ ]:
PAD_IDX = src_vocab.word2idx[PAD_TOKEN]

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = torch.tensor(self.src_vocab.encode(src), dtype=torch.long)
        tgt_ids = torch.tensor(self.tgt_vocab.encode(tgt), dtype=torch.long)
        return src_ids, tgt_ids

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=src_vocab.word2idx[PAD_TOKEN])
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_vocab.word2idx[PAD_TOKEN])
    return src_pad, tgt_pad

BATCH_SIZE = 128
train_loader = DataLoader(TranslationDataset(train_pairs, src_vocab, tgt_vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(TranslationDataset(val_pairs, src_vocab, tgt_vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 5. Arquitectura Transformer (desde cero)

Se implementan manualmente: positional encoding senoidal, atención multi-cabeza (scaled dot-product),
feed-forward posicional, encoder y decoder (con máscara causal para auto-regresión y máscara de padding).

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)


D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT = 256, 8, 512, 3, 0.1
MAX_POS_LEN = MAX_LEN + 2

model = Transformer(len(src_vocab), len(tgt_vocab), d_model=D_MODEL, n_heads=N_HEADS,
                     d_ff=D_FF, n_layers=N_LAYERS, dropout=DROPOUT, max_len=MAX_POS_LEN,
                     pad_idx=PAD_IDX).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params:,}')

## 6. Entrenamiento

Optimizador Adam con label smoothing y *gradient clipping*. Se guarda el checkpoint con menor pérdida
de validación en `transformer_en_es.pth`. Ajusta `N_EPOCHS` según el tiempo disponible en Colab
(con GPU T4, ~1-3 min/época con este tamaño de modelo y dataset).

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
N_EPOCHS = 15  # aumenta si tienes más tiempo/GPU; con 15 ya se ven traducciones razonables
CKPT_PATH = 'transformer_en_es.pth'

best_val = float('inf')
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, model, optimizer)
    val_loss = run_epoch(val_loader, model)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
    dt = time.time() - t0
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | train_loss {train_loss:.3f} | val_loss {val_loss:.3f} | {dt:.1f}s')

print('Mejor val_loss:', best_val)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.xlabel('Época'); plt.ylabel('Loss (CrossEntropy)'); plt.legend(); plt.title('Curva de entrenamiento')
plt.show()

## 7. Inferencia (decodificación greedy)

Se carga el mejor checkpoint y se traduce token a token de forma autoregresiva (greedy: se toma siempre
el token más probable) hasta generar `<eos>` o alcanzar la longitud máxima.

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

def translate_sentence(sentence, model, src_vocab, tgt_vocab, max_len=30):
    model.eval()
    norm = normalize_text(sentence)
    src_ids = torch.tensor([src_vocab.encode(norm)], dtype=torch.long).to(device)
    src_mask = model.make_src_mask(src_ids)
    with torch.no_grad():
        enc_out = model.encoder(src_ids, src_mask)
    tgt_ids = [tgt_vocab.word2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        tgt_mask = model.make_tgt_mask(tgt_tensor)
        with torch.no_grad():
            out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        next_id = out[0, -1].argmax().item()
        tgt_ids.append(next_id)
        if next_id == tgt_vocab.word2idx[EOS_TOKEN]:
            break
    return tgt_vocab.decode(tgt_ids[1:])

for s in ['I love you.', 'What time is it?', 'The weather is nice today.']:
    print(f'{s!r:35s} -> {translate_sentence(s, model, src_vocab, tgt_vocab)}')

## 8. Evaluación en conjunto de test

Se evalúa sobre el conjunto de test completo (≥100 ejemplos, separado desde la sección 3 y nunca visto
durante entrenamiento). La métrica usada es **BLEU** (implementación propia, con *brevity penalty* y
precisión de n-gramas de 1 a 4), estándar para evaluar traducción automática.

In [ ]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def sentence_bleu(reference, hypothesis, max_n=4):
    ref_tokens, hyp_tokens = reference.split(), hypothesis.split()
    if len(hyp_tokens) == 0:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = ngram_counts(ref_tokens, n)
        hyp_ngrams = ngram_counts(hyp_tokens, n)
        overlap = sum(min(c, ref_ngrams.get(g, 0)) for g, c in hyp_ngrams.items())
        total = max(sum(hyp_ngrams.values()), 1)
        precisions.append(overlap / total if total > 0 else 1e-9)
    if min(precisions) <= 0:
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if len(hyp_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    return geo_mean * bp

In [ ]:
results = []
for src, tgt in test_pairs:
    hyp = translate_sentence(src, model, src_vocab, tgt_vocab)
    bleu = sentence_bleu(tgt, hyp)
    results.append({'ingles': src, 'espanol_referencia': tgt, 'espanol_predicho': hyp, 'bleu': bleu})

df = pd.DataFrame(results)
print(f"Tamaño del conjunto de test: {len(df)} ejemplos")
print(f"BLEU promedio en test: {df['bleu'].mean():.4f}")
df.to_csv('resultados_test.csv', index=False)

### Casos donde el modelo traduce bien (BLEU alto)

In [ ]:
pd.set_option('display.max_colwidth', None)
df.sort_values('bleu', ascending=False).head(15)[['ingles','espanol_referencia','espanol_predicho','bleu']]

### Casos donde el modelo falla (BLEU bajo)

In [ ]:
df.sort_values('bleu', ascending=True).head(15)[['ingles','espanol_referencia','espanol_predicho','bleu']]

### Los 100 ejemplos completos del conjunto de test

(quedan también guardados en `resultados_test.csv`, uno de los entregables del proyecto).

In [ ]:
df.head(100)[['ingles','espanol_referencia','espanol_predicho','bleu']]

## 9. Guardado de pesos y vocabularios

Además del checkpoint `transformer_en_es.pth` (guardado automáticamente durante el entrenamiento),
se exportan los vocabularios y la configuración del modelo en JSON, necesarios para el
**notebook de inferencia** (`mt_en_es_inferencia.ipynb`), que puede cargar el modelo sin repetir el entrenamiento.

In [ ]:
with open('vocab_en.json', 'w', encoding='utf-8') as f:
    json.dump(src_vocab.word2idx, f, ensure_ascii=False)
with open('vocab_es.json', 'w', encoding='utf-8') as f:
    json.dump(tgt_vocab.word2idx, f, ensure_ascii=False)

config = {
    'd_model': D_MODEL, 'n_heads': N_HEADS, 'd_ff': D_FF, 'n_layers': N_LAYERS,
    'dropout': DROPOUT, 'max_len': MAX_POS_LEN,
}
with open('model_config.json', 'w') as f:
    json.dump(config, f)

print('Guardado: transformer_en_es.pth, vocab_en.json, vocab_es.json, model_config.json, resultados_test.csv')

## 10. Reporte y conclusiones

### 10.1 Creación del conjunto de entrenamiento
- **Fuente:** corpus paralelo inglés-español de Tatoeba, distribuido por ManyThings.org/Anki
  (`http://www.manythings.org/anki/spa-eng.zip`), descargado automáticamente en la sección 2.
- **Limpieza:** minúsculas, separación de puntuación (`. , ! ? ¿ ¡`) como tokens propios, eliminación de
  caracteres fuera del alfabeto español/inglés, y filtrado de oraciones con más de 20 tokens.
- **Tokenización:** por espacios en blanco sobre el texto normalizado (tokenizador propio, sin BPE ni
  librerías externas).
- **Vocabulario:** construido únicamente sobre el conjunto de entrenamiento, con tokens especiales
  `<pad> <sos> <eos> <unk>` y frecuencia mínima de 2 apariciones para reducir ruido de palabras únicas.
- **Partición:** train / val / test, con al menos 100 pares reservados para validación y otros 100+
  para el conjunto de test final (nunca usado en entrenamiento ni en la elección del mejor checkpoint).

### 10.2 Arquitectura
Transformer encoder-decoder implementado desde cero: positional encoding senoidal, atención multi-cabeza
(8 cabezas), `d_model=256`, `d_ff=512`, 3 capas de encoder y 3 de decoder, dropout 0.1, máscara causal
en el decoder y máscara de padding en ambos. Entrenado con Adam, label smoothing 0.1 y *gradient clipping*.

### 10.3 Resultados
El BLEU promedio obtenido en el conjunto de test (≥100 ejemplos) y ejemplos de traducciones correctas e
incorrectas se muestran en la sección 8. *(Vuelve a ejecutar este notebook en Colab para completar esta
sección con tus números finales antes de entregar el reporte.)*

### 10.4 Limitaciones observadas
- Oraciones largas o con vocabulario poco frecuente en el corpus tienden a producir traducciones
  incompletas o con palabras `<unk>`.
- Al ser un dataset de oraciones cortas y coloquiales (Tatoeba), el modelo generaliza peor a lenguaje
  formal o técnico.
- La decodificación greedy puede quedar atrapada en traducciones subóptimas; *beam search* mejoraría la
  calidad a costa de más cómputo (posible trabajo futuro).